In [ ]:
# get the code
import os
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
# use seaborn plotting defaults
import seaborn as sns; sns.set()

# import package functions (main: sys.path.append('../code') and the modules in code/)
import mycnv
from mycnv.rolling import interpolate, one_col_rolling, llh, rolling_data
from mycnv.rolling_cov import rolling_coverage
from mycnv.rolling_snp import filter_snp   # main imported combineCNV.filter_snp (needs minPONDepth); unused
from mycnv.plot import plot_cov, plot_snp

# main: pd.set_option('mode.chained_assignment', None); no effect with pandas >= 3 (copy-on-write)

# paths and parameters: ../config/config_dev.yaml via dev_config.py (main: hard-coded Dropbox paths)
from dev_config import load_dev
dev = load_dev()
plot_path = dev["plot_path"]
fig_path = dev["fig_path"]
print(f"mycnv {mycnv.__version__} ({os.path.dirname(mycnv.__file__)})")
print(f"sample {dev['sample']} ({dev['study']}) -> {dev['out_dir']}")

# config
### get the config
+ `dev_config.load_dev()` reads `config/config_dev.yaml`: the data paths of the local cache and the Gen-2 CNV params
  (`mycnv.config.load_config`, deep-merged over the mycnv defaults = the somVarWES study config)
+ headless runs: `MYCNV_NB_SAMPLE=<key> jupyter nbconvert --execute ...` (keys under `samples:`)

In [ ]:
CNVconfig = dev["CNVconfig"]

# load raw combined snp data and visualize

In [ ]:
sample = dev["sample"]
# the study's raw/{sample}.snp is plain text (main read notebook 2's output/snp/{sample}.snp.gz)
snp_df = pd.read_csv(dev["raw_snp"], sep="\t")

fig_params = dict(
    figsize=(32,3),
    colormap='coolwarm_r',
    color_chroms=True,
    ylim=(-0,1),
    label_size=13
)

vaf = dict(
        title='BAF',
        plot_type='scatter',  # ['line', 'scatter']
        data='VAF2',
        plot_args=dict(
            s=3,
            color='black',
            cmap='viridis',
            alpha=1
        )
    )
chroms = ['chr3', 'chr4', 'chr5', 'chr6','chr7', 'chr9', 'chr12', 'chr17']
r20 = 'chr20:20.7Mb-34.5Mb'
r7 = 'chr7:95.8Mb-111Mb'
r17 = 'chr17:18.2Mb-25Mb'
fig1, _, _, _ = plot_snp(snp_df, plots=[vaf], chroms="all", region='', **fig_params)
fig, _, df20, _ = plot_snp(snp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig2, _, df7, _ = plot_snp(snp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, df17, _ = plot_snp(snp_df, plots=[vaf], chroms="all", region=r17, **fig_params)
#fig, ax, df, chrom_df

In [ ]:
fig1.savefig(os.path.join(fig_path, f"{sample}_snp_raw.jpeg"))
fig2.savefig(os.path.join(fig_path, f"{sample}_snp_raw.chr7.jpeg"))

# run the code

In [ ]:
from mycnv.rolling_snp import rolling_snp, remerge_cnv

#### load coverage data (output of notebook 3)
roll_cov_df = pd.read_csv(dev["roll_cov"], sep="\t")

# run code
rsnp_df = rolling_snp(snp_df, roll_cov_df, config=CNVconfig)

# visualize
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region='', **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r17, **fig_params)
# cnv_df = remerge_cnv(rsnp_df, roll_cov_df)

In [ ]:
fig1, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region='', **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig2, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r17, **fig_params)
fig1.savefig(os.path.join(fig_path, f"{sample}_snp_clean.jpeg"))
fig2.savefig(os.path.join(fig_path, f"{sample}_snp_clean.chr7.jpeg"))

In [ ]:
cnv_df = remerge_cnv(rsnp_df, roll_cov_df)
cnv_df

In [ ]:
cnv_df.query("VAF1 == VAF1")

### save the data

In [ ]:
rsnp_df.to_csv(dev["cnv_snp"], sep="\t", index=False, compression="gzip")
cnv_df.to_csv(dev["cnv"], sep="\t", index=False, compression="gzip")

# combined functions

In [ ]:
CNVconfig = dev["CNVconfig"]

# load data
sample = dev["sample"]
snp_df = pd.read_csv(dev["raw_snp"], sep="\t")   # plain text
roll_cov_df = pd.read_csv(dev["roll_cov"], sep="\t")

In [ ]:
# canonical code (was an inline copy of code/rollingSNP.py): mycnv.rolling_snp
# names: mergeCov2SNP -> merge_cov_to_snp, get_fallSNP -> get_fall_snp, remergeCNV -> remerge_cnv
# changes vs main (CHANGELOG.md): interpolation and rolling per chromosome, rows in (Chr, FullExonPos) order
from mycnv.rolling_snp import merge_cov_to_snp, center_vaf, make_get_density, get_fall_snp, filter_snp, rolling_snp

In [ ]:
rsnp_df = rolling_snp(snp_df, roll_cov_df, config=CNVconfig)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region='', **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig, _, df7, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, df2, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r17, **fig_params)

In [ ]:
rsnp_df

# Step by step

## SNP data processing
+ merge with coverage data for incorporation of coverage data sets (some data needs interpolation)
+ filtering data for SNPdensity and excessive PONVAF
+  creating ASCAT output

## merge coverage data into SNP
+ reduce to important columns

In [ ]:
roll_cov_df = pd.read_csv(dev["roll_cov"], sep="\t")
roll_cov_df.columns

In [ ]:
from mycnv.rolling_snp import merge_cov_to_snp   # was an inline copy of rollingSNP.mergeCov2SNP

In [ ]:
snp_df = merge_cov_to_snp(snp_df, roll_cov_df)
snp_df.columns

## heteroSNP rolling window
+ #### first, VAF is centered
+ #### next, fallSNP have to be removed
+ #### next, important local features have to be extracted:
    * SNPdensity: measure of genomic spread of SNPs in order to find falling SNPs
    * absVAF: absolute dist measure from meanVAF
    * snpLLH: log-likelihood to belong the center gaussian

+ #### next, these local features have to be computed with a rolling window to convert local to regional data

In [ ]:
from mycnv.rolling_snp import center_vaf   # was an inline copy of rollingSNP.center_vaf

## center VAF

In [ ]:
snp_df['VAF1'].mean()
snp_df['VAF2'].mean()
csnp_df = center_vaf(snp_df)
csnp_df['VAF1'].mean()
csnp_df['VAF2'].mean()

## include metrices to exclude fallSNP
+ compute genomic snp-density ( (window / stretch))
+ (maybe combine with rolling offVAFsum?)
+ remove the negative offVAF in that range

In [ ]:
from mycnv.rolling_snp import make_get_density, get_fall_snp   # inline copies of rollingSNP.make_get_density/get_fallSNP

In [ ]:
exp_df = get_fall_snp(csnp_df, config=CNVconfig)

In [ ]:
dense = dict(
        title='dense',
        plot_type='line',  # ['line', 'scatter']
        data='SNPdensity',
        plot_args=dict(
            color='red',
            alpha=1
        )
    )

fall2 = dict(
        title='fall',
        plot_type='line',  # ['line', 'scatter']
        data='fallSNP',
        plot_args=dict(
            color='green',
            alpha=1
        )
    )

ponvaf = dict(
        title='PONVAF',
        plot_type='line',  # ['line', 'scatter']
        data='PONVAF',
        plot_args=dict(
            color='yellow',
            alpha=1
        )
    )

fig, _, _, _ = plot_snp(exp_df, plots=[vaf, fall2, dense], chroms="all", region='', **fig_params)
fig, _, df20, _ = plot_snp(exp_df, plots=[vaf, fall2, dense], chroms="all", region=r20, **fig_params)
fig, _, df7, _ = plot_snp(exp_df, plots=[vaf, fall2, dense], chroms="all", region=r7, **fig_params)
fig, _, df17, _ = plot_snp(exp_df, plots=[vaf, fall2, dense], chroms="all", region=r17, **fig_params)
fig, _, df17, _ = plot_snp(exp_df, plots=[vaf, fall2, dense], chroms="all", region="chr20", **fig_params)
#fig, ax, df, chrom_df

## filter snp data

In [ ]:
from mycnv.rolling_snp import filter_snp   # was an inline copy of rollingSNP.filter_snp

In [ ]:
CNVconfig = dev["CNVconfig"]
filter_snp_df = filter_snp(exp_df, config=CNVconfig)
fig, _, _, _ = plot_snp(filter_snp_df.query('SNPdensity<0.01'), plots=[vaf], chroms="all", region='', **fig_params)
fig, _, _, _ = plot_snp(filter_snp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig, _, df7, _ = plot_snp(filter_snp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, df2, _ = plot_snp(filter_snp_df, plots=[vaf], chroms="all", region=r17, **fig_params)
len(filter_snp_df.index)

## master function

In [ ]:
from mycnv.rolling_snp import rolling_snp   # was an inline copy of rollingSNP.rolling_snp

In [ ]:
CNVconfig = dev["CNVconfig"]
rsnp_df = rolling_snp(snp_df, roll_cov_df, config=CNVconfig)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region='', **fig_params)
fig, _, _, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r20, **fig_params)
fig, _, df7, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r7, **fig_params)
fig, _, df2, _ = plot_snp(rsnp_df, plots=[vaf], chroms="all", region=r17, **fig_params)

## merge  in the cov data

In [ ]:
rsnp_df.columns

In [ ]:
roll_cov_df.columns

In [ ]:
from mycnv.rolling_snp import remerge_cnv   # was an inline copy of rollingSNP.remergeCNV

In [ ]:
cnv_df = remerge_cnv(rsnp_df, roll_cov_df)
cnv_df

# ######

### LEGACY
+ Gen-1 functions (`remove_fallSNP`, `compute_snp_llh`, `expand_SNPdata`, `rolling_SNP`) with Gen-1 config keys and
  `one_col_rolling` arguments. They are only defined, never called, and do not work against `mycnv`; kept as reference.

In [ ]:
def remove_fallSNP(snp_df, mean=0.5, std=0.2, params={}):
    '''
    removes the falling SNP probably caused by mismapping
    '''
    
    window = params['offVAFwindow']
    cutoff = params['maxFallSNP']
    
    # get the density computer for rolling
    get_SNPdensity = make_get_density(window)
    # cycle through chroms
    chrom_dfs = []
    for chrom in snp_df['Chr'].unique():
        df = snp_df.query('Chr == @chrom')
  
        # get the snp
        df = one_col_rolling(df, df.query('VAF < 0.95'), 'ExonPos', get_SNPdensity, window_size=window, diff_exp=4)
        df.loc[:, 'SNPdensity'] = df['SNPdensity'] / df['SNPdensity'].mean()
    
        # get the offVAFsum
        df = one_col_rolling(df, df.query('VAF < 0.95'), 'offVAF', 'sum', window_size=window, normalize=True, diff_exp=4)
    
        # combine both metrices
        df.loc[:, 'fallSNP'] = df['SNPdensity'] * df['offVAFsum']
        # now remove the ones below average VAFstd
        df = df.query('VAF > @mean - @std / 2 or fallSNP > @cutoff')
        chrom_dfs.append(df)
        
    return pd.concat(chrom_dfs).sort_values('FullExonPos').reset_index(drop=True)

### expand the LLH

In [ ]:
def compute_snp_llh(df, mean=0.5, sigma=0.2):
    '''
    computes the local log-likelihood of belonging to the center gaussian
    '''
        
    show_output(f"Computing log-likelihood of VAF belonging to center gaussian [mean:{round(mean, 3)}, sigma:{round(sigma,3)}]")
    df.loc[:, 'snpLLH'] = llh(df['VAF'], mean, sigma)
    
    # for homoSNPs reduce the VAFs to the ones above mean
    upper_vafs = df.query('@mean < VAF')['VAF']
    # then compute the hsnpLLH

    show_output(f"Computing log-likelihood of VAF belonging to purity100  [mean:1, sigma:{round(sigma,3)}]")
    # these are called hsnp
    # upper_vafs only contains half the snps, the remaining have to be interpolated
    df.loc[:, 'hsnpLLH'] = llh(upper_vafs, 1, sigma)
    df = interpolate(df, 'hsnpLLH', expand_limit=50)
    return df
    

def expand_SNPdata(snp_df, config):
    '''
    retrieve a few data columns locally to use rolling windows on
    this needs to be done chromosome-wise in order to avoid gap effects
    VAF limits are also applied here
    '''
        
    # split the params dict for easier access
    params = config['snp']
    filter_params = params['filter']
    # data_params = params['data']
    
    # reduce the snp_df using lower config limit
    # upper limit has to be set later as we still need the homoSNP llh
    VAFmin, VAFmax = filter_params['VAF']
    snp_df = snp_df.query('@VAFmin < VAF')
    
    
    # get std and mean of VAF
    minVAF, maxVAF = params['LLH']['center_range']
    # get the sigma and mean of the center band VAF (extracted as pd.Series center_vafs)
    center_vafs = snp_df.query('@minVAF < VAF < @maxVAF')['VAF']
    # get width of gaussian from std * sigma_factor
    VAFstd = center_vafs.std()
    VAFmean = center_vafs.mean()
    
    
    # get additional features from VAFs
    snp_df.loc[:, 'offVAF'] = (snp_df['VAF'] - VAFmean) * 2
    # absolute values for cluster 
    snp_df.loc[:,'absVAF'] = np.abs(snp_df['offVAF'])    
    
    ########## remove fallSNP ########
    fs_params = params['fallSNP']
    if fs_params['run']:
        show_output('Removing falling SNPs')
        snp_df = remove_fallSNP(snp_df, mean=VAFmean, std=VAFstd, params=fs_params)
     
    ######## LLH  #####################
    # get the snpLLH and hsnpLLH
    # get config params
    sigma = VAFstd * params['LLH']['sigma_factor']
    # hsnpLLH is computed in order to rescue high absVAF that would have been filtered out
    # lower VAF is already removed because density of VAF ~0 is highly irregular and would confound 
    snp_df = compute_snp_llh(snp_df, mean=VAFmean, sigma=sigma)

    return snp_df.query('VAF < @VAFmax').reset_index(drop=True)

In [ ]:
def rolling_SNP(snp_df, config):
    '''
    cycle through the chroms and perform rolling window computations of snp data set in config
    '''

    # split the params dict for easier access
    params = config['snp']
    filter_params = params['filter']
    data_params = params['rolling_data']
    debug = config['debug']
        
    minDepth = filter_params['minDepth']
    filter_df = snp_df.query('Depth >= @minDepth')
    
    rolling_df = rolling_data(snp_df, filter_df, expand=params['expand'], ddof=config['ddof'], debug=debug, data_params=data_params)
    
    return rolling_df